# Public Holidays – Incremental Load to Lakehouse

This notebook loads public holiday data from the Nager.Date API into a Fabric Lakehouse Delta table using a merge/upsert pattern.

## High-level flow
1. **Configure parameters** – Country, workspace and lakehouse IDs, target table name, and merge key.
2. **Determine incremental window** – Read the existing Delta table with DuckDB to find the latest `date` and compute a lookback range.
3. **Prepare max-date values** – Expose the max date values for downstream logic.
4. **Call API & transform** – Request public holidays for the relevant year, normalize columns, and build a composite key (`Date_Subdivision`).
5. **Merge into Lakehouse** – Upsert rows into the Delta table using the composite key, creating the table on first run.

Use this notebook as a parameterized pattern for incremental API → Lakehouse Delta loads.


### 1. Configuration & global parameters

This cell sets up global configuration used throughout the notebook:
- Imports required Python libraries (`requests`, `pandas`, `datetime`).
- Defines `country_code` (e.g., `"AU"` for Australia).
- Specifies Fabric workspace and lakehouse GUIDs used to build ABFS paths.
- Sets the target Lakehouse Delta table name (`TABLE_NAME = "PublicHolidays"`).
- Defines the merge key (`merge_key = "Date_Subdivision"`) used for upsert logic.
- Declares date column names and the lookback window (`LOOKBACK_MONTHS`) to control incremental loading.


In [5]:
import requests
import pandas as pd
from datetime import datetime

# Variables
country_code = "AU"         # AU = Australia

# Get the workspace GUID which contains the Lakehouse you want to read/write
workspace_guid = "16961f8e-3218-4723-8e6e-6684eb47688a"

# Get the Lakehouse GUID
lakehouse_guid = "5397e662-ae12-4fd8-b5c4-fcab0ccd9baf"

# Lakehouse table name to load
TABLE_NAME = "PublicHolidays"

# Merge key used to insert/update rows in the Delta table
merge_key = "Date_Subdivision"

# Date columns used by the load
# Incremental loads are now based on column below.
Lakehouse_Date_Column = "date"
API_Date_Filter_Column = "date"

# Reload window used by the existing process
LOOKBACK_MONTHS = 2



### 2. Read existing Delta table & derive incremental range

This cell:
- Imports `duckdb`, `pandas`, and `notebookutils`.
- Builds the full ABFS path to the target Delta table (`TABLE_ABFS_PATH`).
- Retrieves a storage access token from Fabric.
- Configures a DuckDB connection with a Fabric OneLake secret.
- Queries the existing Delta table via `delta_scan` to compute:
  - `MaxDate` – the maximum value in the Lakehouse date column.
  - `MaxDate_Date` – `MaxDate` cast to `DATE` for safe comparisons.
- Handles missing/invalid dates and, if needed, falls back to a fixed start date.
- Computes `RANGE_START` = `MaxDate_Date - LOOKBACK_MONTHS`, which is used as the start of the incremental load window.


In [6]:
from datetime import datetime
import duckdb
import pandas as pd
import notebookutils

# ============================================
# Configuration
# ============================================

# Build full ABFS path
TABLE_ABFS_PATH = f"abfss://{workspace_guid}@onelake.dfs.fabric.microsoft.com/{lakehouse_guid}/Tables/{TABLE_NAME}"

# ============================================
# Get Storage Token (required for DuckDB in Fabric)
# ============================================
storage_token = notebookutils.credentials.getToken('storage')

# Create DuckDB connection with storage options
con = duckdb.connect()

# Register the Delta Lake table using ABFS path
con.execute(f"""
    SET home_directory = '';
    CREATE OR REPLACE SECRET onelake_secret (
        TYPE AZURE,
        PROVIDER ACCESS_TOKEN,
        ACCESS_TOKEN '{storage_token}'
    );
""")

# ============================================
# Get Max approvedAt (handles None values) - best for incremental loads
# ============================================
df_MaxDate = con.execute(f"""
    SELECT 
        MAX({Lakehouse_Date_Column}) AS MaxDate,
        MAX(CAST({Lakehouse_Date_Column} AS DATE)) AS MaxDate_Date
    FROM delta_scan('{TABLE_ABFS_PATH}')
    WHERE {Lakehouse_Date_Column} IS NOT NULL 
      AND {Lakehouse_Date_Column} <> 'None' 
      AND {Lakehouse_Date_Column} <> ''
""").fetchdf()

# ============================================
# Convert to clean Python datetime
# ============================================
if not df_MaxDate.empty and df_MaxDate['MaxDate_Date'].notna().any():
    max_date_raw = df_MaxDate['MaxDate_Date'].max()
    max_date_dt = pd.to_datetime(max_date_raw).normalize().to_pydatetime()
    
    # Apply lookback (go back X months from the max date)
    RANGE_START = max_date_dt - pd.DateOffset(months=LOOKBACK_MONTHS)
    
    print(f"✅ Last {Lakehouse_Date_Column} in Lakehouse     : {max_date_dt}")
    print(f"✅ RANGE_START (max - {LOOKBACK_MONTHS} month(s)) : {RANGE_START}")
    print(f"Type: {type(RANGE_START)}")
else:
    # Fallback if table is empty or no dates
    RANGE_START = datetime(2025, 10, 1)
    print(f"⚠️ No valid {Lakehouse_Date_Column} found. Using fallback RANGE_START = {RANGE_START}")

print(RANGE_START)


✅ Last date in Lakehouse     : 2026-12-26 00:00:00
✅ RANGE_START (max - 2 month(s)) : 2026-10-26 00:00:00
Type: <class 'pandas._libs.tslibs.timestamps.Timestamp'>
2026-10-26 00:00:00


### 3. Expose max-date values for downstream logic

This cell:
- Extracts the maximum timestamp (`max_date`) and date-only (`max_date_date`) values from `df_MaxDate`.
- Uses `display(max_date_date)` to surface the effective max date in the Lakehouse for debugging and validation.
- Provides `max_date` for use as the `input_date` in the API call configuration.


In [7]:
# Get Max Dates to Delete and Max Date to Load

max_date = df_MaxDate['MaxDate'].max() if not df_MaxDate.empty else None
max_date_date = df_MaxDate['MaxDate_Date'].max() if not df_MaxDate.empty else None

display(max_date_date)


Timestamp('2026-12-26 00:00:00')

### 4. Call Nager.Date API, transform data & merge into Lakehouse

This cell contains the main business logic:

1. **Configuration**
   - Sets `input_date` from `max_date` obtained earlier.
   - Builds the Lakehouse Delta table ABFS path (`LAKEHOUSE_ABFS_PATH`).

2. **API call & transformation – `get_public_holidays`**
   - Derives the year from `input_date` and calls the Nager.Date `/Holidays/{country}/{year}` endpoint.
   - Converts the JSON response to a `pandas` DataFrame.
   - Ensures `date` is a proper datetime column.
   - Creates `SubdivisionCodeText` to normalize the `subdivisionCodes` list/nullable field.
   - Builds a composite key `Date_Subdivision` using date (`YYYYMMDD`) and subdivision code text.
   - Casts all columns to string for Lakehouse consistency.

3. **Merge into Delta – `merge_to_lakehouse`**
   - Retrieves a Fabric storage token and configures Delta-RS storage options.
   - Attempts to open the existing Delta table at `table_path`.
   - Reads existing rows and computes:
     - `insert_count` – new keys not present in the Lakehouse.
     - `update_count` – keys that already exist and will be updated.
   - Executes a Delta merge (`WHEN MATCHED THEN UPDATE`, `WHEN NOT MATCHED THEN INSERT`).
   - If the table does not exist or merge fails, creates a new Delta table via `write_deltalake`.

4. **Main orchestration**
   - Calls `get_public_holidays(input_date, country_code)`.
   - Displays the resulting DataFrame for inspection.
   - If there are rows, calls `merge_to_lakehouse` to upsert them into the Lakehouse table; otherwise prints a message that no records were returned.


In [8]:
import requests
import pandas as pd
from datetime import datetime
from deltalake import DeltaTable, write_deltalake
import pyarrow as pa
import notebookutils

# ============================================
# Configuration
# ============================================

# Get the Last Loaded Date
input_date = max_date #"2026-07-07"

# This is the Lakehouse Table Path using ABFS
LAKEHOUSE_ABFS_PATH = (
    f"abfss://{workspace_guid}@onelake.dfs.fabric.microsoft.com/"
    f"{lakehouse_guid}/Tables/{TABLE_NAME}"
)

# ============================================
# API Call
# ============================================

def get_public_holidays(input_date: str, country_code: str) -> pd.DataFrame:
    year = datetime.strptime(input_date, "%Y-%m-%d").year

    url = f"https://date.nager.at/api/v4/Holidays/{country_code}/{year}"

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data)

    if df.empty:
        return df

    df["date"] = pd.to_datetime(df["date"])

    # Handle subdivisionCodes being null or list
    df["SubdivisionCodeText"] = df["subdivisionCodes"].apply(
        lambda x: "ALL" if x is None else "-".join(x) if isinstance(x, list) else str(x)
    )

    # Composite key
    df["Date_Subdivision"] = (
        df["date"].dt.strftime("%Y%m%d") + "_" + df["SubdivisionCodeText"]
    )

    # Convert all columns to string for Lakehouse consistency
    df = df.astype(str)

    return df

# ============================================
# Merge to Lakehouse
# ============================================

def merge_to_lakehouse(new_df: pd.DataFrame, table_path: str, merge_key: str):

    access_token = notebookutils.credentials.getToken("storage")

    storage_options = {
        "bearer_token": access_token,
        "use_fabric_endpoint": "true"
    }

    table = pa.Table.from_pandas(new_df, preserve_index=False)

    try:
        dt = DeltaTable(table_path, storage_options=storage_options)

        existing_df = dt.to_pyarrow_table().to_pandas()

        # Count Inserts vs Updates
        existing_keys = set(existing_df[merge_key])
        new_keys = set(new_df[merge_key])

        insert_count = len(new_keys - existing_keys)
        update_count = len(new_keys & existing_keys)

        print("=" * 60)
        print(f"Rows received : {len(new_df):,}")
        print(f"Rows to insert: {insert_count:,}")
        print(f"Rows to update: {update_count:,}")
        print("=" * 60)

        dt.merge(
            source=table,
            predicate=f"target.{merge_key}=source.{merge_key}",
            source_alias="source",
            target_alias="target"
        ).when_matched_update_all() \
         .when_not_matched_insert_all() \
         .execute()

        final_rows = dt.to_pyarrow_table().num_rows

        print()
        print("=" * 60)
        print("MERGE COMPLETE")
        print(f"Inserted : {insert_count:,}")
        print(f"Updated  : {update_count:,}")
        print(f"Total Rows : {final_rows:,}")
        print("=" * 60)

    except Exception:

        print("Table does not exist. Creating it...")

        write_deltalake(
            table_path,
            table,
            mode="overwrite",
            storage_options=storage_options
        )

        print("=" * 60)
        print("NEW TABLE CREATED")
        print(f"Inserted : {len(new_df):,}")
        print("Updated  : 0")
        print("=" * 60)

        

    except Exception as e:
        print(f"Table not found or merge failed: {e}")
        print("Creating new Delta table...")

        write_deltalake(
            table_path,
            table,
            mode="overwrite",
            storage_options=storage_options
        )

        print("✅ New Delta table created")

# ============================================
# Main
# ============================================

df = get_public_holidays(input_date, country_code)

display(df)

if not df.empty:
    merge_to_lakehouse(df, LAKEHOUSE_ABFS_PATH, merge_key)
else:
    print("No records returned from API")

Rows received : 27
Rows to insert: 0
Rows to update: 27

MERGE COMPLETE
Inserted : 0
Updated  : 27
Total Rows : 27
